# Week 5: Data Engineering for Noisy Sensor Data
## Missing data, outliers, normalisation and splits without leakage

**Course:** AI for Aerospace & Mechanical Engineering (PhD) &nbsp;|&nbsp; **Block 1:** Classical ML

> **Provenance:** partly expanded from instructor notes, partly new content. The framing in
> Section 0 (classical models such as decision trees, SVMs, naive Bayes and logistic regression
> cannot learn from a raw CSV; a preprocessing step must turn raw data into features) expands
> slides 2 to 8 of the instructor's deck *Machine Learning & Decision Trees* (which the instructor
> adapted from the Faradars blog). Everything else is new content: missing-data mechanisms follow
> Little and Rubin, *Statistical Analysis with Missing Data* (2019), Ch. 1; cleaning, imputation,
> outliers, scaling and leakage are rewritten (not copied) from W3Schools ("Pandas - Cleaning
> Data", "Python Machine Learning Scaling", "Train/Test") and GeeksforGeeks ("Handling Missing
> Values in Machine Learning", "Detect and Remove the Outliers using Python", "Data Leakage");
> the LiDAR error types from Thrun, Burgard and Fox, *Probabilistic Robotics* (2005), Sec. 6.3;
> the Hampel filter from Pearson (2002); blocked validation from Roberts et al., *Ecography* (2017).

**How to use this notebook.** Run it top to bottom. Cells marked `# TODO` are your exercises;
each is followed by a check that prints `[x]` when your answer is right and `[ ]` while it is
not done. The solved copy is `week05_data_engineering_solved.ipynb`.

## 0. The problem: a braking controller that trusts a raw log

The final-project rover (module M3) decides when to brake from the **distance to the obstacle
one second from now**. We have 16 recorded test drives: in each, the rover waits, accelerates
towards a wall, brakes and stops about a metre in front of it. The logger writes one row every
0.1 s: run number, time, forward LiDAR range, wheel speed, longitudinal acceleration and yaw rate.

The task is a regression: predict `range_ahead_m`, the LiDAR range 1.0 s later, from the four
sensor values now. The instructor's deck puts it in one sentence: classical models "cannot be
applied directly to raw data such as CSV files"; a preprocessing step must come first. Let us see
what happens if we skip it: read the CSV, shift the range by ten rows to get the target, drop
the NaN rows, split at random and fit two Week 4 strategies.

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path
from typing import Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

ROVER_CSV: Final[Path] = REPO_ROOT / "data" / "sample" / "week05_rover_log.csv"
SEED: Final[int] = 5            # seed of the lecture log
HORIZON_STEPS: Final[int] = 10  # 1.0 s at 10 Hz


def check(name: str, value: object, expected: float, tol: float) -> None:
    """Print a pass mark if ``value`` is within ``tol`` of ``expected``.

    A value of None means the exercise has not been attempted yet.
    """
    if value is None:
        print(f"[ ] {name}: not done yet")
        return
    assert abs(float(value) - expected) <= tol, f"{name}: got {value}, expected ~{expected}"
    print(f"[x] {name} = {float(value):.6g}")


print("Repository root found:", REPO_ROOT.name)

In [ ]:
from aerospace_ai.models import KNearestNeighbors, LinearLeastSquares, MeanBaseline
from aerospace_ai.splits import GroupHoldoutSplit, RandomRowSplit

FEATURES: Final[list[str]] = ["lidar_range_m", "wheel_speed_mps", "accel_x_mps2", "gyro_z_dps"]
TARGET: Final[str] = "range_ahead_m"

raw = pd.read_csv(ROVER_CSV)
naive = raw.copy()
naive[TARGET] = naive["lidar_range_m"].shift(-HORIZON_STEPS)   # deliberately careless
naive = naive.dropna()
X_naive, y_naive = naive[FEATURES].to_numpy(), naive[TARGET].to_numpy()
(random_fold,) = RandomRowSplit(0.25, seed=0).split(len(naive))

naive_rmse = {}
for model in (MeanBaseline(), LinearLeastSquares(), KNearestNeighbors(k=5)):
    model.fit(X_naive[random_fold.train], y_naive[random_fold.train])
    naive_rmse[model.name] = model.score(X_naive[random_fold.test], y_naive[random_fold.test])
    print(f"{model!r:<40} test RMSE {naive_rmse[model.name]:.3f} m")

An error of 2.7 m on a 1 m stopping distance would put the rover into the wall. Worse, the two
numbers above are not even honest: the random split is leaky (Section 6). By the end of this
notebook the same linear model reaches about **0.12 m** on test drives it has never seen, and
we will be able to say *why* we trust that number. Four things are wrong with the naive cell:

| Defect (Week 5 name) | Where it hides | Section |
|---|---|---|
| sentinels and missing values | LiDAR `0.0` = "no echo"; NaN wheel speeds | 1, 2 |
| outliers | dust echoes, accelerometer spikes, a frozen gyroscope | 3 |
| boundaries | the shift takes targets from the *next* run; duplicated rows | 4 |
| leakage | random split of a time series; statistics from all rows | 5, 6 |

## 1. Look before you clean: the data-quality audit

Never clean a table you have not described. `audit_frame` (in `aerospace_ai.preprocessing`)
counts NaN, ±inf and sentinel cells per column, the longest run of NaN and the **longest run of
exactly repeated values** (a frozen sensor repeats itself; real noisy signals do not).

In [ ]:
from aerospace_ai.preprocessing import audit_frame, count_duplicate_keys

audit = audit_frame(raw, sentinels={"lidar_range_m": 0.0})
display(audit.round(3))
print("rows:", len(raw), "| runs:", raw["run_id"].nunique())

Read the table like an engineer, column by column:

* `lidar_range_m`: 533 zeros (21 %). A LiDAR that receives no echo reports 0.0. Some of these
  are random dropouts, most come in long blocks at the start of a run (66 in a row).
* `wheel_speed_mps`: 19 NaN, in short bursts (CAN-bus packet loss). Its 51 repeats are the
  rover standing still at exactly 0.0: genuine, not a defect.
* `accel_x_mps2`: min and max of about ±25 m/s² (2.5 g) on a rover that brakes at 1 m/s².
* `gyro_z_dps`: 32 identical values in a row. With 0.3 °/s of noise this cannot be real.

**Exercise 1.1.** Loggers re-send packets. Count the rows whose `(run_id, time_s)` repeats an
earlier row with `count_duplicate_keys`.

In [ ]:
n_duplicates = None   # TODO: count_duplicate_keys(raw, ("run_id", "time_s"))
check("Exercise 1.1 duplicated rows", n_duplicates, 25, 0)

In [ ]:
from aerospace_ai.rover import RoverLogConfig, RoverLogGenerator

truth = RoverLogGenerator(RoverLogConfig(seed=SEED)).generate_truth()   # synthetic: truth is known
run_raw, run_true = raw[raw["run_id"] == 3], truth[truth["run_id"] == 3]

fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
for ax, col, unit in zip(axes, FEATURES, ["m", "m/s", "m/s²", "°/s"]):
    ax.plot(run_true["time_s"], run_true[col], color="0.6", lw=2, label="truth")
    ax.plot(run_raw["time_s"], run_raw[col], ".", ms=3, color="C3", label="logged")
    ax.set_ylabel(f"{col}\n[{unit}]", fontsize=8)
    ax.grid(alpha=0.3)
axes[0].axhline(12.0, ls="--", color="k", lw=0.8)
axes[0].text(0.2, 12.3, "LiDAR max range 12 m", fontsize=8)
axes[0].legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("time [s]")
fig.suptitle("Run 3: what the logger wrote vs what happened")
fig.tight_layout()

## 2. Missing data: why it is missing decides what to do

Three mechanisms (Little and Rubin; GeeksforGeeks "Handling Missing Values"):

| Mechanism | Meaning | Rover example | Safe remedies |
|---|---|---|---|
| **MCAR** missing completely at random | unrelated to anything | a dropped CAN packet | drop rows, interpolate, impute |
| **MAR** missing at random | depends on *other observed* values | GPS lost when the logged pitch is steep | impute *using* those values |
| **MNAR** missing not at random | depends on the *missing value itself* | LiDAR silent **because** the wall is beyond 12 m | none is neutral: model it, flag it, or restrict the scope |

### Worked Example 1 (by hand): mean imputation under MNAR

Eight true ranges along one run: 2, 4, 6, 8, 10, 13, 15, 17 m. The LiDAR cannot see beyond 12 m,
so the last three are missing. The true mean is $75/8 = 9.375$ m; the mean of what we observed is
$30/5 = 6.0$ m. Filling the gaps with 6.0 m makes the three far readings wrong by 7, 9 and 11 m
and pulls the mean **down** by 3.4 m: the imputer has invented a wall where there is none. A
missing-value indicator ("LiDAR saw nothing", 1/0) keeps the real information, which is
*"farther than 12 m"*.

In [ ]:
true8 = np.array([2.0, 4.0, 6.0, 8.0, 10.0, 13.0, 15.0, 17.0])
seen = true8[true8 <= 12.0]
imputed = np.where(true8 <= 12.0, true8, seen.mean())
print(f"true mean {true8.mean():.3f} | observed mean {seen.mean():.3f} | after imputation {imputed.mean():.3f}")
print("imputation errors on the far readings:", (imputed - true8)[true8 > 12.0])

log = raw.drop_duplicates(["run_id", "time_s"]).reset_index(drop=True)   # one row per sample
far = truth["lidar_range_m"].to_numpy() > 12.0
print(f"\nrover log: true mean {truth['lidar_range_m'].mean():.2f} m, "
      f"true mean of the out-of-range samples {truth['lidar_range_m'][far].mean():.2f} m")

**Exercise 2.1.** In the de-duplicated `log`, compute the mean of the LiDAR readings that are
*not* 0.0. Compare it with the true mean printed above (6.86 m). Which way is it biased, and why
must it be?

In [ ]:
observed_mean = None   # TODO: mean of log["lidar_range_m"] where the value is not 0.0
check("Exercise 2.1 observed LiDAR mean [m]", observed_mean, 5.091, 0.001)

### From sentinel to NaN, then decide per mechanism

`MarkInvalid` turns the sentinel 0.0, ±inf and impossible values (outside the LiDAR's 0.3 to
12 m specification) into NaN: it says "unknown" instead of "touching the sensor". Then:

* **short interior gaps** (MCAR dropouts, ≤ 5 samples): interpolate in time (`FillGaps`, Week 3);
* **leading blocks** (MNAR, wall beyond range): we *restrict the model's scope* to "wall within
  sensor range" and drop those rows; Assignment 2 counts the two kinds separately;
* **long gaps in a weak feature** (frozen gyroscope): fill with the training median *after*
  the split (Section 5).

### Worked Example 2 (by hand): linear interpolation of a gap

The LiDAR reads 8.0 m at $t = 0$ and 7.4 m at $t = 0.3$ s; the readings at 0.1 and 0.2 s are
missing. Linear interpolation assumes constant speed across the gap:
$r(t) = 8.0 + (7.4 - 8.0)\,t/0.3$, so $r(0.1) = 7.8$ m and $r(0.2) = 7.6$ m. The implied speed is
2 m/s; at 10 Hz and rover speeds below 2.2 m/s, a 0.5 s gap (5 samples) stays within a few
centimetres of the truth. Longer gaps are left as NaN: interpolating across them invents data.

In [ ]:
from aerospace_ai.pipeline import FillGaps
from aerospace_ai.preprocessing import MarkInvalid

gap = pd.DataFrame({"time_s": [0.0, 0.1, 0.2, 0.3], "lidar_range_m": [8.0, 0.0, 0.0, 7.4]})
gap = MarkInvalid(("lidar_range_m",), sentinels=(0.0,), valid_min=0.3, valid_max=12.0)(gap)
filled = FillGaps(("lidar_range_m",), max_gap=5, time_column="time_s")(gap)
display(pd.concat([gap, filled["lidar_range_m"].rename("filled")], axis=1))
check("Worked Example 2 r(0.2 s)", filled["lidar_range_m"].iloc[2], 7.6, 1e-9)

## 3. Outliers: global rules fail, local rules work

### Worked Example 3 (by hand): the z-score cannot see one outlier in a small sample

Accelerometer samples at rest: 0.2, −0.1, 0.0, 0.3, −0.2, 0.1, 0.0 and one glitch of 25.0 m/s².
Mean $= 25.3/8 = 3.1625$; population standard deviation $= 8.255$. The glitch has
$z = (25 - 3.1625)/8.255 = 2.645 < 3$: **not flagged**. This is no accident. With $n$ samples, no $|z|$ computed with the population
standard deviation can exceed $\sqrt{n-1}$ (Samuelson's inequality, 1968): 2.646 for $n = 8$,
so our glitch is almost as extreme as a $z$-score can be, and 3 is reachable only from
$n = 10$ on. (With the sample standard deviation the bound is $(n-1)/\sqrt n$; Shiffler, 1988.)
The outlier inflates the very standard deviation that
should expose it (*masking*). The median (0.05) and the MAD (0.15) barely notice it: the robust
score $|25 - 0.05|/(1.4826 \cdot 0.15) = 112$ flags it at once.

In [ ]:
glitch = np.array([0.2, -0.1, 0.0, 0.3, -0.2, 0.1, 0.0, 25.0])
z = (glitch - glitch.mean()) / glitch.std()
median = np.median(glitch)
mad = np.median(np.abs(glitch - median))
robust = np.abs(glitch - median) / (1.4826 * mad)
print(f"z of the glitch {z[-1]:.3f} | robust score {robust[-1]:.1f}")
check("Worked Example 3 z-score of the glitch", z[-1], 2.645, 1e-3)

**Exercise 3.1.** Evaluate the masking bound $\sqrt{n-1}$ for $n = 8$, and find the smallest
$n$ for which a single outlier *can* reach $z = 3$.

In [ ]:
bound_8 = None      # TODO: sqrt(n - 1) for n = 8
smallest_n = None   # TODO: smallest integer n with sqrt(n - 1) >= 3
check("Exercise 3.1 bound for n = 8", bound_8, 2.6458, 1e-4)
check("Exercise 3.1 smallest n", smallest_n, 10, 0)

### Three tools, three kinds of outlier

* **Global z-score** ($|z| > 3$): only for roughly Gaussian data, large samples, no trend.
* **Tukey's fences** $[Q_1 - k\,\mathrm{IQR},\ Q_3 + k\,\mathrm{IQR}]$, $k = 1.5$ ("outside") or
  $k = 3$ ("far out"): robust, but global. On the rover, braking at −1.2 m/s² lies *outside*
  the $k = 1.5$ fences of the whole log: a real manoeuvre, not an error. Assignment 2 uses them
  as a last safety net with $k = 3$, fitted on the training split.
* **Hampel filter** (Week 3): median and MAD in a moving window. It compares each sample with its
  *neighbours*, so a dust echo of 1.2 m while the wall is 8 m away stands out even though 1.2 m
  is a perfectly valid range a few seconds later.

In [ ]:
from aerospace_ai.pipeline import HampelFilter

accel = log["accel_x_mps2"]
z_all = (accel - accel.mean()) / accel.std(ddof=0)
q1, q3 = accel.quantile([0.25, 0.75])
tukey = (accel < q1 - 1.5 * (q3 - q1)) | (accel > q3 + 1.5 * (q3 - q1))
hampel_flags, _ = HampelFilter(("accel_x_mps2",)).outlier_mask(accel)
big = accel.abs() > 5.0   # 0.5 g: certainly not this rover
print(f"|a| > 5 m/s^2: {int(big.sum())} | z > 3: {int((z_all.abs() > 3).sum())} | "
      f"Tukey k=1.5: {int(tukey.sum())} | Hampel: {int(hampel_flags.sum())}")
print("Tukey flags", int((tukey & ~big).sum()), "plausible samples (braking and noise)")

## 4. Respect the boundaries: per-run processing

The log is 16 separate drives glued together. Any operation that looks at neighbours
(interpolation, Hampel windows, rolling means, the target shift) must stop at the end of a run.
In the naive cell, the last ten rows of each run took their "future range" from the **start of
the next run**, 10 m away. `PerGroup` is a composite stage: it applies any stateless Week 3 stage
to each run separately, without editing that stage.

In [ ]:
from aerospace_ai.preprocessing import ForecastTarget, PerGroup, SortAndDeduplicate

shift = ForecastTarget("lidar_range_m", HORIZON_STEPS, TARGET)
end_of_run_1 = log[log["run_id"] == 1].index[-1]
print("plain shift, last row of run 1:   ", shift(log)[TARGET].iloc[end_of_run_1])
print("per-run shift, last row of run 1: ", PerGroup(shift, "run_id")(log)[TARGET].iloc[end_of_run_1])

### The lecture recipe

`aerospace_ai.datasets.rover_cleaning_pipeline` chains everything so far. It contains **only
stateless stages**, so it may run on the whole log before the split. `run` records what every
stage did (Week 3).

In [ ]:
from aerospace_ai.datasets import rover_cleaning_pipeline

cleaning = rover_cleaning_pipeline(HORIZON_STEPS)
print(cleaning)
result = cleaning.run(raw)
clean = result.output
display(result.report_frame()[["stage", "rows_in", "rows_out", "nan_out", "cells_changed"]])
print("rows kept:", len(clean), "of", len(raw))

The Hampel step changed 167 cells, although only 22 were dust echoes or spikes (11 + 11). The
others are noisy samples nudged towards their local median: by about 4 cm (at most 7 cm) for
the range and about 0.3 m/s² for the acceleration. That is the price of a robust filter: small
next to what it removes, but not zero, so report it.

## 5. Normalisation, and the rule that keeps it honest

Distances and gradients see raw units. Range spans 0.3 to 12 m, acceleration ±1.2 m/s², yaw
rate ±4 °/s: unscaled, the range dominates every k-NN distance (Week 4 showed the same with
altitude). Three scalers (W3Schools "ML Scaling"; GeeksforGeeks):

| Scaler | Formula | Use when |
|---|---|---|
| z-score (`Standardize`, Week 3) | $(x - \mu)/\sigma$ | roughly Gaussian, no outliers |
| min-max (`MinMaxScale`) | $(x - x_{\min})/(x_{\max} - x_{\min})$ | bounded signals, image pixels |
| robust (`RobustScale`) | $(x - \tilde x)/\mathrm{IQR}$ | heavy tails or remaining spikes |

### Worked Example 4 (by hand): which neighbour is nearest?

Query $q$ = (range 5.0 m, accel 0.0: cruising). Candidate A = (5.1 m, −1.0 m/s²: braking),
candidate B = (6.2 m, 0.0: cruising). Raw distances: $d_A = \sqrt{0.1^2 + 1.0^2} = 1.005$,
$d_B = 1.200$, so the **braking** sample A is "nearest": one m/s² counts as much as one metre.
Divide by training standard deviations of 3.0 m and 0.4 m/s²: $d_A = \sqrt{0.033^2 + 2.5^2} =
2.500$, $d_B = 1.2/3 = 0.400$, and the cruising sample B is nearest. Which neighbour wins must
not depend on the units we happened to log in.

**The rule.** A scaler, an imputer or a clipper *learns* numbers. Learn them from the training
rows only, then apply the same numbers to validation and test rows (GeeksforGeeks "Data
Leakage"). Learned stages in the course library subclass `ColumnStatStage` and refuse to
transform before `fit`.

In [ ]:
from aerospace_ai.preprocessing import ColumnStatStage, ImputeMedian, MinMaxScale, RobustScale

scaled_q = np.array([5.0 / 3.0, 0.0])
for name, point in {"A": (5.1, -1.0), "B": (6.2, 0.0)}.items():
    raw_d = math.dist((5.0, 0.0), point)
    scaled_d = math.dist(scaled_q, (point[0] / 3.0, point[1] / 0.4))
    print(f"{name}: raw distance {raw_d:.3f}, scaled distance {scaled_d:.3f}")
print("learned stages share one template:", [c.__name__ for c in ColumnStatStage.__subclasses__()])

**Exercise 5.1.** Split the cleaned table by run (`GroupHoldoutSplit(0.25)`: runs 1 to 12 train,
13 to 16 test). Fit `RobustScale` on the four features of the **training** frame and read the
learned median of `lidar_range_m` from `stats_`. Then fit a second scaler on *all* rows and
compare: the difference is the information that would leak from the test drives.

In [ ]:
from aerospace_ai.splits import split_frame

((train_frame, test_frame),) = split_frame(clean, GroupHoldoutSplit(0.25), "run_id")
scaler = None          # TODO: RobustScale over FEATURES, fitted on train_frame
train_median = None    # TODO: scaler.stats_["lidar_range_m"][0]
check("Exercise 5.1 training median of the range [m]", train_median, 5.076, 0.001)

## 6. Splits without leakage

Neighbouring samples at 10 Hz are almost twins. A random row split puts the sample at 4.1 s in
the test set and its twins at 4.0 and 4.2 s in the training set; k-NN then *looks up* the answer.
`aerospace_ai.splits` offers four strategies (Week 4 Strategy pattern) and `check_split`, which
reports shared rows, shared runs and the closest train/test neighbours:

| Splitter | Test set | Leak-free for this log? |
|---|---|---|
| `RandomRowSplit` | random 25 % of rows | no: test rows sit between training rows |
| `ChronologicalSplit(gap=0)` | last 25 % of rows | almost: one run is cut, its two halves touch |
| `ChronologicalSplit(gap=10)` | same, 10 rows purged | yes: no training target comes from the test period |
| `GroupHoldoutSplit` | last 25 % of runs | yes, and it asks the real question: a *new* drive |

In [ ]:
from aerospace_ai.datasets import DatasetFactory
from aerospace_ai.splits import ChronologicalSplit, check_split, split_dataset

data = DatasetFactory.load("rover_range_forecast", path=ROVER_CSV)   # same recipe, as a Dataset
data = data.subset(np.isfinite(data.X).all(axis=1), "finite")       # all four features finite
print(data)

splitters = [RandomRowSplit(0.25, seed=0), ChronologicalSplit(0.25), ChronologicalSplit(0.25, gap=10),
             GroupHoldoutSplit(0.25)]
rows = []
for splitter in splitters:
    (fold,) = splitter.split(len(data), data.groups)
    ((train, test),) = split_dataset(data, splitter)
    report = check_split(fold, data.groups, horizon=HORIZON_STEPS)
    row = {"splitter": repr(splitter), "leak check": "OK" if report.ok else "LEAKY"}
    for model in (LinearLeastSquares(), KNearestNeighbors(k=5)):
        row[model.name] = model.fit(train.X, train.y).score(test.X, test.y)
    rows.append(row)
split_table = pd.DataFrame(rows).set_index("splitter").round(3)
split_table

The linear model barely cares (it has four parameters and cannot memorise), but k-NN looks
**twice as good** with the random split as on new drives. Had we tuned k on the random split,
we would have reported 0.32 m and delivered 0.64 m. The honest choice for "will it work on the
next drive?" is the group split.

**Exercise 6.1.** Read the k-NN test RMSE under `GroupHoldoutSplit` from `split_table`.

In [ ]:
knn_group_rmse = None   # TODO: split_table.loc[<group splitter row>, "k_nearest_neighbors"]
check("Exercise 6.1 k-NN RMSE on new drives [m]", knn_group_rmse, 0.641, 0.001)

### Two more leaks to recognise on sight

* **Non-causal features.** Week 3's `RollingMean` is *centred*: the smoothed value at $t$ uses
  samples up to $t + w/2$. For a forecasting model that is a peek at the future.
  `CausalRollingMean` uses only past samples and could run on the rover.
* **Time-series cross-validation.** `ExpandingWindowCV` trains on blocks 1..k and tests on block
  k + 1, never the reverse; with `gap` it purges the rows whose targets overlap the test block.
  Week 7's `GridSearchCV` will take such a splitter.

In [ ]:
from aerospace_ai.pipeline import RollingMean
from aerospace_ai.preprocessing import CausalRollingMean

step = pd.DataFrame({"x": [0.0, 0.0, 0.0, 0.0, 9.0, 9.0]})
both = pd.concat([RollingMean(("x",), window=3)(step)["x"].rename("centred"),
                  CausalRollingMean(("x",), window=3)(step)["x_trailing_mean"].rename("trailing")], axis=1)
display(both.T)
print("The centred mean reacts at index 3, one sample BEFORE the step happens.")

## 7. Putting it together: the leak-free workflow

1. **Stateless cleaning** on the whole log (Section 4 recipe).
2. **Split** by run.
3. **Fit** the learned stages on the training rows: median imputation of the gyroscope (the
   frozen segments become NaN in Assignment 2), robust scaling of all features.
4. **Transform** training and test rows with the *same* fitted pipeline.
5. Fit and score the model.

In [ ]:
from aerospace_ai.pipeline import DataPipeline

learned = DataPipeline([ImputeMedian(("gyro_z_dps",)), RobustScale(tuple(FEATURES))])
learned.fit(train_frame)                                  # training rows only
X_train = learned.transform(train_frame)[FEATURES].to_numpy()
X_test = learned.transform(test_frame)[FEATURES].to_numpy()
y_train, y_test = train_frame[TARGET].to_numpy(), test_frame[TARGET].to_numpy()

final = {}
for model in (MeanBaseline(), LinearLeastSquares(), KNearestNeighbors(k=5)):
    final[model.name] = model.fit(X_train, y_train).score(X_test, y_test)
comparison = pd.DataFrame({"naive (random split)": naive_rmse, "clean (group split)": final}).round(3)
comparison

In [ ]:
model = LinearLeastSquares().fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(9, 3.5))
for run_id in test_frame["run_id"].unique()[:2]:
    rows = (test_frame["run_id"] == run_id).to_numpy()
    t = test_frame.loc[rows, "time_s"]
    ax.plot(t, y_test[rows], color="0.5", lw=2, label=f"run {run_id}: measured range in 1 s")
    ax.plot(t, model.predict(X_test[rows]), "--", label=f"run {run_id}: linear forecast")
ax.set_xlabel("time [s]")
ax.set_ylabel("range in 1 s [m]")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
print("weights on scaled features:", dict(zip(FEATURES, model.coef_.round(3))))

The forecast follows the wall down to rest. The weights say what physics says: the future range
is the present range minus speed × 1 s; acceleration adds a little, the yaw rate almost nothing.

## 8. Final checks

This cell verifies the worked examples and the library behaviour. It must pass on every machine
(this is the notebook's own test cell); it does not depend on the exercises.

In [ ]:
assert math.isclose(true8.mean() - imputed.mean(), 3.375, abs_tol=1e-9)    # MNAR bias
assert math.isclose(filled["lidar_range_m"].iloc[1], 7.8, abs_tol=1e-9)
assert z[-1] < 3.0 < robust[-1]
assert naive_rmse["linear_least_squares"] > 2.0                 # the raw log is useless...
assert final["linear_least_squares"] < 0.15                      # ...the clean one is not
assert split_table["k_nearest_neighbors"].iloc[0] < 0.6 * split_table["k_nearest_neighbors"].iloc[-1]
assert list(split_table["leak check"]) == ["LEAKY", "LEAKY", "OK", "OK"]
assert not set(train_frame["run_id"]) & set(test_frame["run_id"])
assert learned.is_fitted
print("All Week 5 checks passed.")

## What we now know how to do

* Audit a raw sensor log before touching it: sentinels, NaN, repeats, duplicates, ranges.
* Name the missing-data mechanism (MCAR, MAR, MNAR) and choose deletion, interpolation,
  imputation with an indicator, or a restricted model scope accordingly.
* Explain why a global z-score misses outliers in small samples and use Tukey fences or a
  Hampel filter instead; tell a spike from a manoeuvre.
* Keep every neighbour-based operation inside one run with a composite `PerGroup` stage.
* Scale features with z-score, min-max or robust scalers whose statistics come from the
  training split only.
* Split time-ordered, grouped data without leakage (chronological with a gap, by run,
  expanding-window CV) and prove it with `check_split`.